In [18]:
# 风险及免责提示：该策略由聚宽用户在聚宽社区分享，仅供学习交流使用。
# 原文一般包含策略说明，如有疑问请到原文和作者交流讨论。
# 原文网址：https://www.joinquant.com/view/community/detail/37446
# 标题：【策略.行业轮动】趋势-拥挤-景气度模型看行业轮动
# 作者：Wowo_QI
import os
import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta
from dateutil.relativedelta import *
import time
import math
import sqlite3
from tqdm import tqdm

import talib
import statsmodels.api as sm
from jqdatasdk import *
# auth('XXXXXXXXXX','XXXXX')

import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams["font.family"] = 'Arial Unicode MS' 
plt.rcParams['axes.unicode_minus'] = False # 用来正常显示负号
from matplotlib.font_manager import FontProperties
myfont=FontProperties(fname=r'C:\Windows\Fonts\simhei.ttf',size=14)

import pyecharts.options as opts
from pyecharts.charts import Kline, Line, Page, Grid

from IPython import display
display.set_matplotlib_formats('svg') # 提高绘图清晰度

import seaborn as sns
sns.set(font=myfont.get_name())

# 將html另存爲圖片jpeg
from pyecharts.render import make_snapshot
from snapshot_phantomjs import snapshot # js法
# import pyecharts # cromedriver法
# from snapshot_selenium import snapshot

import warnings
warnings.filterwarnings("ignore")

auth success 


## 限定分析范围

In [19]:
ystd = str(datetime.now().date()+timedelta(days=-1))
today = str(datetime.now().date())
info_dt = ystd
rect_2_trd_dt = [str(dt) for dt in get_trade_days(end_date=info_dt, count=2)] # 注：本脚本只有交易日判断会用到聚宽数据！
info_dt_m1 = rect_2_trd_dt[-1] if info_dt not in rect_2_trd_dt else rect_2_trd_dt[-2]

# info_dt_m1 = '2022-04-20' # 补数据（注意数据库最早日期！！）
print('最近一个交易日（非当天）为：'+info_dt_m1)

最近一个交易日（非当天）为：2022-04-22


In [3]:
# 获取股票池列表
df_stk_pool = pd.read_excel('../../basic_auto_get_factors/df_far_{info_dt_m1}.xlsx'.format(info_dt_m1=info_dt_m1)) # 注：依赖当日已经跑出的因子数据！
df_ind_stk = df_stk_pool[['code','name','sw_l1_name']]
stk_list = list(df_ind_stk['code'])
df_stk_pool.head(3)

,code,trade_date,name,EPttm,BPttm,SPttm,NCFPttm,OCFP,PEG,PEadj_G,...,turn_240d,bias_turn_240d,std_FF3factor_20d,std_FF3factor_60d,std_FF3factor_120d,std_FF3factor_240d,sw_l1_code,sw_l1_name,sw_l2_code,sw_l2_name
0,000001.XSHE,2022-04-20,平安银行,0.118133,1.058313,0.550691,-0.174679,-0.626603,0.330556,0.331319,...,0.516755,-0.073862,0.002252,0.001066,-0.000090,0.000086,801780,银行I,801783,股份制银行II
1,000002.XSHE,2022-04-20,万科A,0.100128,1.048878,2.012882,-0.199840,0.018285,NaN,NaN,...,1.052373,0.103703,0.002053,0.000615,0.001917,0.001046,801180,房地产I,801181,房地产开发II
2,000063.XSHE,2022-04-20,中兴通讯,0.064135,0.484637,1.078051,0.072180,0.148017,0.318128,0.146575,...,1.452239,-0.162434,0.005225,-0.003637,-0.001579,-0.000699,801770,通信I,801102,通信设备II


In [4]:
# 获取近一年个股股价变动及近三年个股换手率
day_m1y = str(int(info_dt_m1.split('-')[0])-1)+'-'+info_dt_m1.split('-')[1]+'-'+info_dt_m1.split('-')[2]
day_m4y = str(int(info_dt_m1.split('-')[0])-4)+'-'+info_dt_m1.split('-')[1]+'-'+info_dt_m1.split('-')[2]
conn = sqlite3.connect('../../basic_auto_get_factors/factors_base_Price_and_TAindex.db')
print ("因子数据库（价格及技术指标）打开成功")
c = conn.cursor()
sql_ = '''SELECT day, code, chg_close
        FROM stocks_chg_close_data 
        WHERE day>='{day_m4y}' AND day<='{info_dt_m1}'
        '''.format(day_m4y=day_m4y,info_dt_m1=info_dt_m1)
p_chg = pd.read_sql(sql_,conn)
p_chg = p_chg.loc[p_chg['code'].isin(stk_list)==True]
conn.commit()
conn.close()
conn = sqlite3.connect('../../basic_auto_get_factors/factors_base_valuation.db')
print ("因子数据库（市值及估值）打开成功")
c = conn.cursor()
sql_ = '''SELECT day, code, market_cap, turnover_ratio
        FROM stocks_valuation_data
        WHERE day>='{day_m4y}' AND day<='{info_dt_m1}'
        '''.format(day_m4y=day_m4y,info_dt_m1=info_dt_m1)
val = pd.read_sql(sql_,conn)
val = val.loc[val['code'].isin(stk_list)==True]
conn.commit()
conn.close()



因子数据库（价格及技术指标）打开成功
因子数据库（市值及估值）打开成功


## 通用计算


In [5]:
## 构建运算大表
p_chg = pd.merge(p_chg,val,how='left',on=['code','day'])
p_chg = pd.merge(p_chg, df_stk_pool[['code','sw_l1_name']],how='left',on='code')
p_chg.head(3)

,day,code,chg_close,market_cap,turnover_ratio,sw_l1_name
0,2018-04-20,000001.XSHE,-0.010092,1948.8417,0.5667,银行I
1,2018-04-23,000001.XSHE,0.019462,1986.6166,0.6326,银行I
2,2018-04-24,000001.XSHE,0.025455,2036.4108,0.8636,银行I


In [6]:
## 计算全市场收益序列 -- 市值加权（非行业等权）
day_list = list(set(p_chg['day']))
day_list.sort(reverse=False)
df_mkt_r = pd.DataFrame()
for dt in tqdm(day_list):
    df_dt = p_chg.loc[p_chg['day']==dt]
    r_mkt = (df_dt['chg_close']*df_dt['market_cap']).sum()/(df_dt['market_cap'].sum())
    df_mkt_r = df_mkt_r.append({'date':dt,'r_mkt':r_mkt},ignore_index=True)
df_mkt_r.head(3)

100%|████████████████████████████████████████████████████████████████████████████████| 972/972 [01:51<00:00,  8.74it/s]


,date,r_mkt
0,2018-04-20,-0.014485
1,2018-04-23,-0.001848
2,2018-04-24,0.021591


## 行业动量（趋势度）

In [7]:
## 计算行业动量
day_list = list(set(p_chg.loc[p_chg['day']>=day_m1y]['day'])) # 确定时间范围为过去1年
day_list.sort(reverse=False)

# 计算各行业IR值（行业动量）
ind_list = list(set(p_chg['sw_l1_name']))
df_ind_mom = pd.DataFrame()
for ind_ in tqdm(ind_list):
    df_ind_r = pd.DataFrame()
    df_ind = p_chg.loc[p_chg['sw_l1_name']==ind_]
    for dt in day_list:
        df_dt = df_ind.loc[df_ind['day']==dt]
        r_ind = (df_dt['chg_close']*df_dt['market_cap']).sum()/(df_dt['market_cap'].sum())
        df_ind_r = df_ind_r.append({'date':dt,'r_ind':r_ind},ignore_index=True)
    df_mgd = pd.merge(df_ind_r, df_mkt_r, how='left', on='date')
    r_ind_1y = (df_mgd['r_ind']+1).cumprod().iloc[-1]-1
    r_mkt_1y = (df_mgd['r_mkt']+1).cumprod().iloc[-1]-1
    error_std = (df_mgd['r_ind']-df_mgd['r_mkt']).std()
    IR_ind = (r_ind_1y-r_mkt_1y)/error_std
    df_ind_mom = df_ind_mom.append({'ind_name':ind_,
                                    'r_ind_1y':r_ind_1y, 'r_mkt_1y':r_mkt_1y, 'error_std':error_std,
                                    'IR_ind':IR_ind},ignore_index=True)
    
print(df_ind_mom.sort_values(by='IR_ind',ascending=False).head(3))

100%|██████████████████████████████████████████████████████████████████████████████████| 31/31 [00:44<00:00,  1.45s/it]


,ind_name,r_ind_1y,r_mkt_1y,error_std,IR_ind
26,煤炭I,0.836483,0.044286,0.026720,29.648406
29,化工I,0.288449,0.044286,0.011567,21.108217
10,电气设备I,0.412199,0.044286,0.019247,19.115572
18,建筑装饰I,0.291038,0.044286,0.014631,16.864651
30,有色金属I,0.336211,0.044286,0.018697,15.613184
13,综合I,0.257188,0.044286,0.015209,13.998233
1,公用事业I,0.206701,0.044286,0.014099,11.519350
8,交通运输I,0.115309,0.044286,0.009768,7.270865
20,石油石化I,0.150288,0.044286,0.015612,6.789714
4,钢铁I,0.141457,0.044286,0.019322,5.029152


## 换手率/波动率/beta均值（拥挤度）

In [37]:
# 计算个股3个月（40个交易日）换手率、波动率、beta
print('========== 计算个股3个月（40个交易日）换手率、波动率、beta ==========')
df_cpt_per_stk = pd.DataFrame()
for stk in tqdm(stk_list):
#     print(stk) # tmp检查
    df_stk_cpted = pd.DataFrame()
    df_stk = p_chg.loc[p_chg['code']==stk].sort_values(by=['day'],ascending=True)
    df_stk_cpted['day'], df_stk_cpted['code'], df_stk_cpted['sw_l1_name'] = df_stk['day'], df_stk['code'], df_stk['sw_l1_name']
    # 波动率
    df_stk_cpted['p_chg_40std'] = [np.nan]*39+[df_stk.iloc[i:i+40]['chg_close'].std() for i in range(len(df_stk)-39)]
    # 换手率
    df_stk_cpted['tovr_MA40'] = [np.nan]*39+[df_stk.iloc[i:i+40]['turnover_ratio'].mean() for i in range(len(df_stk)-39)]
    # beta
    beta_ = [np.nan]*39
    for i in range(len(df_stk)-39):
        x_y = pd.merge(df_stk.iloc[i:i+40],df_mkt_r,how='left',left_on='day',right_on='date').dropna()
        if len(x_y)<40:
            beta_.append(np.nan)
        else:
            y = x_y['chg_close'].values
            x = x_y['r_mkt'].values
            X = sm.add_constant(x)
            model = sm.OLS(y,X,missing='drop')
            results = model.fit()
            beta_.append(results.params[1])
    df_stk_cpted['beta_40d'] = beta_
    df_cpt_per_stk = df_cpt_per_stk.append(df_stk_cpted.dropna(), ignore_index=True)

# 计算行业各指标等权平均值
print('========== 计算行业各指标等权平均值 ==========')
ind_list = list(set(df_cpt_per_stk['sw_l1_name']))
day_list = list(set(df_cpt_per_stk['day']))
day_list.sort(reverse=False)
df_cpt_per_ind = pd.DataFrame()
for ind_ in tqdm(ind_list):
    df_ind_cpted = pd.DataFrame()
    df_ind = df_cpt_per_stk.loc[df_cpt_per_stk['sw_l1_name']==ind_]
    df_ind_cpted['day'], df_ind_cpted['sw_l1_name'] = day_list, [ind_]*len(day_list)
    std_list, tovr_list, beta_list = [], [], []
    for dt in day_list:
        avg_tmp = df_ind.loc[df_ind['day']==dt].mean()
        std_list.append(avg_tmp['p_chg_40std'])
        tovr_list.append(avg_tmp['tovr_MA40'])
        beta_list.append(avg_tmp['beta_40d'])
    df_ind_cpted['p_chg_40std'], df_ind_cpted['tovr_MA40'], df_ind_cpted['beta_40d'] = std_list, tovr_list, beta_list
    df_cpt_per_ind = df_cpt_per_ind.append(df_ind_cpted, ignore_index=True)
    
# 计算行业各指标同全行业均值之比
print('========== 计算行业各指标同全行业均值之比 ==========')
df_cpt_per_ind_2Avg = pd.DataFrame()
for dt in tqdm(day_list):
    df_dt_ind = df_cpt_per_ind.loc[df_cpt_per_ind['day']==dt]
    avg_tmp = df_dt_ind.mean()
    df_dt_ind['p_chg_40std_2Avg'] = df_dt_ind['p_chg_40std']/avg_tmp['p_chg_40std']
    df_dt_ind['tovr_MA40_2Avg'] = df_dt_ind['tovr_MA40']/avg_tmp['tovr_MA40']
    df_dt_ind['beta_40d_2Avg'] = df_dt_ind['beta_40d']/avg_tmp['beta_40d']
    df_dt_ind = df_dt_ind[['day','sw_l1_name','p_chg_40std_2Avg','tovr_MA40_2Avg','beta_40d_2Avg']]
    df_cpt_per_ind_2Avg = df_cpt_per_ind_2Avg.append(df_dt_ind, ignore_index=True)
    
# 计算各行业滚动N年的z-score
print('========== 计算各行业滚动N年的z-score ==========')
df_ind_crowd = pd.DataFrame()
for ind_ in tqdm(ind_list):
    df_ratio_ind = df_cpt_per_ind_2Avg.loc[df_cpt_per_ind_2Avg['sw_l1_name']==ind_].sort_values(by='day',ascending=True)
    z_p_chg_40std = (df_ratio_ind.iloc[-1]['p_chg_40std_2Avg']-df_ratio_ind['p_chg_40std_2Avg'].mean())/(df_ratio_ind['p_chg_40std_2Avg'].std())
    z_tovr_MA40 = (df_ratio_ind.iloc[-1]['tovr_MA40_2Avg']-df_ratio_ind['tovr_MA40_2Avg'].mean())/(df_ratio_ind['tovr_MA40_2Avg'].std())
    z_beta_40d = (df_ratio_ind.iloc[-1]['beta_40d_2Avg']-df_ratio_ind['beta_40d_2Avg'].mean())/(df_ratio_ind['beta_40d_2Avg'].std())
    crowdedness = (z_p_chg_40std+z_tovr_MA40+z_beta_40d)/3
    df_ind_crowd = df_ind_crowd.append({'ind_name':ind_, 
                                        'z_p_chg_40std':z_p_chg_40std,'z_tovr_MA40':z_tovr_MA40,'z_beta_40d':z_beta_40d,
                                        'crowdedness':crowdedness},ignore_index=True)
print(df_ind_crowd.sort_values(by='crowdedness',ascending=False).head(3))

,ind_name,z_p_chg_40std,z_tovr_MA40,z_beta_40d,crowdedness
25,房地产I,4.871941,6.803625,1.926271,4.533946
18,建筑装饰I,1.989586,3.388530,-0.402666,1.658483
2,休闲服务I,0.998844,1.465906,1.868227,1.444325
8,交通运输I,2.462377,-0.076909,1.151762,1.179077
13,综合I,1.716610,2.313245,-0.605228,1.141542
3,医药生物I,1.206693,1.677172,0.042823,0.975563
26,煤炭I,1.137642,1.434022,0.035463,0.869042
20,石油石化I,0.917690,1.737051,-0.140861,0.837960
4,钢铁I,0.445555,1.068541,0.739940,0.751346
14,建筑材料I,0.555141,0.964873,0.620093,0.713369


## 营收、净利、ROE增长（景气度）
对于各行业，三者取行业各股中位数（防止亏损、盈利大幅波动带来的影响）

In [33]:
df_g_ix = df_stk_pool[['code','trade_date','sw_l1_name','sales_G_ttm','profit_G_ttm','ocf_G_ttm','roe_G_ttm']]
ind_list = list(set(df_g_ix['sw_l1_name']))
df_ind_boom = pd.DataFrame()
for ind_ in ind_list:
    df_ind = df_g_ix.loc[df_g_ix['sw_l1_name']==ind_]
    median_tmp = df_ind.quantile(0.5)
    boom_ = (median_tmp['sales_G_ttm']+median_tmp['profit_G_ttm']+median_tmp['ocf_G_ttm']+median_tmp['roe_G_ttm'])/4
    df_ind_boom = df_ind_boom.append({'ind_name':ind_,
                                      'sales_G_ttm':median_tmp['sales_G_ttm'],'profit_G_ttm':median_tmp['profit_G_ttm'],
                                      'ocf_G_ttm':median_tmp['ocf_G_ttm'],'roe_G_ttm':median_tmp['roe_G_ttm'],
                                      'boom':boom_},
                                     ignore_index=True)
print(df_ind_boom.sort_values(by='boom',ascending=False).head(3))

,ind_name,sales_G_ttm,profit_G_ttm,ocf_G_ttm,roe_G_ttm,boom
26,煤炭I,0.330022,1.373143,1.129491,1.185922,1.004644
4,钢铁I,0.386420,1.139944,0.821124,0.814484,0.790493
30,有色金属I,0.398634,1.007697,0.321648,0.793635,0.630404
29,化工I,0.309827,0.792362,0.231530,0.512852,0.461643
23,商业贸易I,0.062840,0.426110,0.425746,0.241945,0.289160
27,美容护理I,0.159113,0.248554,0.543436,0.076791,0.256973
24,国防军工I,0.188086,0.333504,0.392768,0.112794,0.256788
19,电子I,0.328420,0.392880,0.086292,0.146726,0.238579
8,交通运输I,0.243330,0.225712,0.356209,0.104987,0.232559
20,石油石化I,0.269131,0.165467,0.366092,0.089042,0.222433


## 合并数据、储存、绘图

In [36]:
# 合并数据并储存
df_mgd = df_ind_mom.copy()
df_mgd = pd.merge(df_mgd, df_ind_crowd, how='left', on='ind_name')
df_mgd = pd.merge(df_mgd, df_ind_boom, how='left', on='ind_name')
df_mgd['trade_date'] = [info_dt_m1]*len(df_mgd)
df_mgd.to_excel('industry_3d_alys_{info_dt_m1}.xlsx'.format(info_dt_m1=info_dt_m1),index=False)
print(df_mgd.head(3))

  ind_name  r_ind_1y  r_mkt_1y  error_std     IR_ind  z_p_chg_40std  \
0     计算机I -0.141340  0.044286   0.009334 -19.887187      -1.537470   
1    公用事业I  0.206701  0.044286   0.014099  11.519350       0.185740   
2    休闲服务I -0.339163  0.044286   0.014015 -27.359806       0.998844   

   z_tovr_MA40  z_beta_40d  crowdedness  sales_G_ttm  profit_G_ttm  ocf_G_ttm  \
0    -0.072758   -0.958909    -0.856379     0.157199      0.012121  -0.350528   
1    -0.152758    1.242213     0.425065     0.191124      0.024722  -0.046211   
2     1.465906    1.868227     1.444325     0.164942     -0.108970   0.279701   

   roe_G_ttm      boom  trade_date  
0  -0.116296 -0.074376  2022-04-20  
1  -0.144060  0.006394  2022-04-20  
2  -0.278958  0.014179  2022-04-20  


In [3]:
# 绘图
df_mgd = pd.read_excel('industry_3d_alys_{info_dt_m1}.xlsx'.format(info_dt_m1=info_dt_m1))
df_plot = df_mgd[['ind_name','IR_ind','crowdedness','boom']]
df_plot['IR_ind'] = (df_plot['IR_ind']-df_plot['IR_ind'].mean())/df_plot['IR_ind'].std()
df_plot['crowdedness'] = (df_plot['crowdedness']-df_plot['crowdedness'].mean())/df_plot['crowdedness'].std()
df_plot['boom'] = (df_plot['boom']-df_plot['boom'].mean())/df_plot['boom'].std()/20
df_plot['tol'] = 0.51*df_plot['IR_ind'] - 0.23*df_plot['crowdedness'] + 1.13*df_plot['boom']
print(df_plot.sort_values(by=['tol'],ascending=False).head(3))

   ind_name    IR_ind  crowdedness      boom       tol
26      煤炭I  2.168223     0.659003  0.152780  1.126865
29      化工I  1.620508    -0.388142  0.057187  0.980352
10    电气设备I  1.492711    -0.528560  0.006012  0.889645


In [1]:
from pyecharts import options as opts
from pyecharts.charts import Scatter
from pyecharts.commons.utils import JsCode

c = (
    Scatter(init_opts=opts.InitOpts(
            width="1000px",
            height="900px",
            animation_opts=opts.AnimationOpts(animation=False),
            bg_color='white')
           )
    .add_xaxis(df_plot['IR_ind'])
    .add_yaxis("景气度(大小)    综合得分(颜色)",
               [list(z) for z in zip(df_plot['crowdedness'], df_plot['boom'], df_plot['tol'], df_plot['ind_name'])],
               label_opts=opts.LabelOpts(
                   formatter=JsCode(
                       "function(params){return params.value[4];}"
                   )
               ),
              )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="行业趋势-拥挤-景气模型"),
        visualmap_opts=[opts.VisualMapOpts(type_="size",
                                           pos_top = 50,
                                           max_=df_plot['boom'].max(), min_=df_plot['boom'].min(),
                                           dimension=2),
                        opts.VisualMapOpts(type_="color",
                                           pos_bottom = 50,
                                           max_=df_plot['tol'].max(), min_=df_plot['tol'].min(),
                                           dimension=3)
                       ],
        yaxis_opts=opts.AxisOpts(name="拥挤度",splitline_opts=opts.SplitLineOpts(is_show=True)),
        xaxis_opts=opts.AxisOpts(name="趋势度",splitline_opts=opts.SplitLineOpts(is_show=True)),
    )
)
c.render("scatter_trend_crowded_boom.html")
make_snapshot(snapshot,c.render('scatter_trend_crowded_boom.HTML'),
              'scatter_trend_crowded_boom.jpeg',is_remove_html=True)
# jupyter展示
c.render_notebook()